In [ ]:
import os
import json
from datasets import load_dataset
from transformers import DetrForObjectDetection, DetrFeatureExtractor, TrainingArguments, Trainer
import torch
from transformers import pipeline
import matplotlib.pyplot as plt
from PIL import Image

# Disable W&B logging using --report_to flag
os.environ["WANDB_DISABLED"] = "true"

# Step 1: Define the paths
data_dir = '/content/data'  # Path to your data directory
images_dir = os.path.join(data_dir, 'images')
metadata_path = os.path.join(data_dir, 'metadata.jsonl')
coco_path = os.path.join(data_dir, 'result.json')  # Path to the COCO export result.json

# Step 2: Generate metadata.jsonl from result.json if it doesn't exist
if not os.path.exists(metadata_path):
    print(f"{metadata_path} not found. Generating metadata.jsonl from result.json...")
    
    if os.path.exists(coco_path):
        # Load COCO formatted annotations
        with open(coco_path, 'r') as f:
            cocodata = json.load(f)

        # Initialize an empty list to store Huggingface formatted data
        huggingdata = []

        # Convert the COCO annotations to Huggingface format
        for image in cocodata['images']:
            image['file_name'] = image['file_name'].split(os.path.sep)[-1]
            image['image_id'] = image['id']
            image['objects'] = {'bbox': [], 'category': [], 'area': [], 'id': []}
            
            # Match annotations to the images
            for annot in cocodata['annotations']:
                if annot['image_id'] == image['id']:
                    image['objects']['bbox'].append(annot['bbox'])
                    image['objects']['category'].append(annot['category_id'])
                    image['objects']['area'].append(annot['area'])
                    image['objects']['id'].append(annot['id'])
            
            huggingdata.append(image)

        # Write the Huggingface formatted data to a jsonl file
        with open(metadata_path, 'w') as f:
            for item in huggingdata:
                f.write(json.dumps(item) + "\n")

        print("metadata.jsonl generated successfully!")
    else:
        print(f"Error: {coco_path} not found. Cannot generate metadata.jsonl.")
else:
    print(f"{metadata_path} already exists. Skipping generation.")

# Step 3: Load the dataset
try:
    candy_data = load_dataset('imagefolder', data_dir=data_dir)
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset: {str(e)}")
    candy_data = None  # Ensure candy_data is set to None if dataset loading fails

# Only proceed with training if dataset was loaded successfully
if candy_data is not None:
    # Step 4: Split the dataset into training and testing sets
    split_candy_data = candy_data['train'].train_test_split(test_size=0.2)

    # Step 5: Load the pre-trained model and feature extractor
    model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
    feature_extractor = DetrFeatureExtractor.from_pretrained("facebook/detr-resnet-50")

    # Step 6: Dynamically list image files and preprocess the dataset
    image_files = os.listdir(images_dir)  # List all images in the folder

    def preprocess(example, idx):
        # Dynamically get the correct image path based on index and images folder contents
        image_file = os.path.join(images_dir, image_files[idx])  # Match index with file list
        
        # Open the image and preprocess it
        image = Image.open(image_file)
        encoded_image = feature_extractor(images=image, return_tensors="pt")
        example['pixel_values'] = encoded_image['pixel_values'][0]  # Ensure pixel_values are tensors
        
        # Convert bounding box and category into the format needed for labels
        example['labels'] = {
            'image_id': torch.tensor(example['image_id'], dtype=torch.int64),
            'boxes': torch.tensor(example['objects']['bbox'], dtype=torch.float32),  # Change 'bbox' to 'boxes'
            'class_labels': torch.tensor(example['objects']['category'], dtype=torch.int64),
            'area': torch.tensor(example['objects']['area'], dtype=torch.float32)
        }
        return example

    # Apply preprocessing to the train and test sets
    split_candy_data = split_candy_data.map(preprocess, with_indices=True)

    # Step 7: Define training arguments
    training_args = TrainingArguments(
        output_dir='./results',
        per_device_train_batch_size=2,
        num_train_epochs=10,
        logging_steps=10,
        save_steps=10,
        save_total_limit=2,
        evaluation_strategy="epoch",
        remove_unused_columns=False,  # Prevent automatic removal of columns
        report_to="none"  # Ensure W&B logging is fully disabled
    )

    # Step 8: Define data collator and trainer
    def collate_fn(batch):
        # Ensure pixel_values are tensors for stacking
        pixel_values = torch.stack([torch.tensor(item["pixel_values"]) for item in batch])
        labels = [{k: torch.tensor(v) for k, v in t["labels"].items()} for t in batch]
        return {"pixel_values": pixel_values, "labels": labels}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=split_candy_data["train"],
        eval_dataset=split_candy_data["test"],
        data_collator=collate_fn
    )

    # Step 9: Train the model
    trainer.train()

    # Step 10: Evaluate the model
    eval_results = trainer.evaluate()
    print(f"Evaluation Results: {eval_results}")

    # Step 11: Save the trained model
    trainer.save_model("candy_detector")

    # Step 12: Create the inference function for candy counting
    obj_detector = pipeline("object-detection", model="candy_detector")

    def candy_counter(image):
        detections = obj_detector(image, threshold=0.5)
        
        # Initialize candy type counts
        candy_counts = {
            'Moon': 0, 'Insect': 0, 'Black_star': 0, 'Grey_star': 0,
            'Unicorn_whole': 0, 'Unicorn_head': 0, 'Owl': 0, 'Cat': 0
        }
        
        # Mapping from category IDs to candy labels
        id2label = {1: 'Moon', 2: 'Insect', 3: 'Black_star', 4: 'Grey_star',
                    5: 'Unicorn_whole', 6: 'Unicorn_head', 7: 'Owl', 8: 'Cat'}
        
        # Count detected objects
        for detection in detections:
            label_id = detection['label']
            label_name = id2label.get(label_id, None)
            if label_name:
                candy_counts[label_name] += 1
        
        return candy_counts

    # Function to plot bounding boxes
    def plot_bounding_boxes(image, detections):
        plt.imshow(image)
        for detection in detections:
            bbox = detection['box']
            plt.gca().add_patch(plt.Rectangle((bbox['xmin'], bbox['ymin']),
                                              bbox['xmax'] - bbox['xmin'],
                                              bbox['ymax'] - bbox['ymin'],
                                              edgecolor='red', facecolor='none', lw=2))
        plt.show()

    # Step 13: Test the candy counter with an example image
    test_image_path = os.path.join(images_dir, 'your_test_image.jpg')  # Replace with your actual test image name
    image = Image.open(test_image_path)
    plt.imshow(image)
    plt.show()

    # Run the candy_counter function on the test image
    detections = obj_detector(image, threshold=0.5)
    result = candy_counter(image)
    print(result)

    # Plot the bounding boxes
    plot_bounding_boxes(image, detections)

else:
    print("Dataset loading failed. Cannot proceed with training.")